# KFP Pipeline: Fraud Detection (TrainerV2 + Feast + KServe)

This pipeline orchestrates the full MLOps workflow using **Kubeflow Pipelines**:

| Step | Component | What it does |
|------|-----------|-------------|
| 1 | **Submit TrainJob** | Submits a Kubeflow TrainerV2 TrainJob that fetches from Feast, trains, uploads model to MinIO |
| 2 | **Quality Gate** | Reads metrics.json from MinIO, enforces minimum AUC threshold |
| 3 | **Deploy KServe** | Creates/patches a KServe InferenceService (conditional on quality gate) |

The TrainJob does ALL the heavy lifting: Feast fetch, split, train, evaluate, export sklearn model, upload to MinIO.
The pipeline just orchestrates and gates deployment.

## 1) Install dependencies

In [ ]:
!pip install -q kfp kubeflow

## 2) Define pipeline components

In [ ]:
from kfp import dsl

### Component 1 — Submit TrainerV2 TrainJob

Uses `kubeflow.trainer.TrainerClient` to submit a TrainJob. The training function
(defined inline) fetches features from Feast remote offline store, trains a PyTorch FraudMLP,
exports an sklearn-compatible model.joblib, and uploads everything to MinIO.

This is the SAME function we tested in `run_training_feast_minio.py`.

In [ ]:
@dsl.component(
    base_image="registry.access.redhat.com/ubi9/python-311:latest",
    packages_to_install=["kubeflow", "kubernetes"],
)
def submit_train_job(
    workshop_ns: str,
    feast_start_date: str,
    feast_end_date: str,
    num_epochs: int,
    batch_size: int,
    learning_rate: float,
    hidden_dim: int,
    val_split: float,
    minio_endpoint: str,
    minio_access_key: str,
    minio_secret_key: str,
    minio_bucket: str,
    minio_model_prefix: str,
):
    """Submit a Kubeflow TrainerV2 TrainJob that fetches from Feast, trains, uploads to MinIO."""
    import textwrap
    from kubeflow.trainer import CustomTrainer, TrainerClient

    def train_fraud_from_feast(
        num_epochs=5, batch_size=256, lr=1e-3, hidden_dim=64,
        feast_registry_url=None, feast_offline_host=None, feast_offline_port=8815,
        feast_start_date="2025-01-01", feast_end_date="2025-03-31",
        feast_features=None, label_column="fraud", val_split=0.2,
        output_dir="/tmp/model_output",
        minio_endpoint="http://minio-service.kubeflow.svc.cluster.local:9000",
        minio_access_key="minio", minio_secret_key="minio123",
        minio_bucket="models", minio_model_prefix="fraud-detector",
        workshop_ns=None,
    ):
        import json, os, random
        import numpy as np, pandas as pd, torch
        import torch.distributed as dist
        from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
        from sklearn.model_selection import train_test_split
        from torch import nn
        from torch.utils.data import DataLoader, Dataset, DistributedSampler

        if isinstance(num_epochs, dict):
            cfg = num_epochs
            num_epochs = cfg.get('num_epochs', 5); batch_size = cfg.get('batch_size', 256)
            lr = cfg.get('lr', 1e-3); hidden_dim = cfg.get('hidden_dim', 64)
            feast_registry_url = cfg.get('feast_registry_url'); feast_offline_host = cfg.get('feast_offline_host')
            feast_offline_port = cfg.get('feast_offline_port', 8815)
            feast_start_date = cfg.get('feast_start_date', feast_start_date)
            feast_end_date = cfg.get('feast_end_date', feast_end_date)
            feast_features = cfg.get('feast_features'); label_column = cfg.get('label_column', 'fraud')
            val_split = cfg.get('val_split', 0.2); output_dir = cfg.get('output_dir', output_dir)
            minio_endpoint = cfg.get('minio_endpoint', minio_endpoint)
            minio_access_key = cfg.get('minio_access_key', minio_access_key)
            minio_secret_key = cfg.get('minio_secret_key', minio_secret_key)
            minio_bucket = cfg.get('minio_bucket', minio_bucket)
            minio_model_prefix = cfg.get('minio_model_prefix', minio_model_prefix)
            workshop_ns = cfg.get('workshop_ns', workshop_ns)

        ns = workshop_ns or os.environ.get('WORKSHOP_NS', 'mlops-workshop')
        if feast_registry_url is None: feast_registry_url = f'feast-registry-service.{ns}.svc.cluster.local:6567'
        if feast_offline_host is None: feast_offline_host = f'feast-offline-service.{ns}.svc.cluster.local'
        if minio_model_prefix == 'fraud-detector': minio_model_prefix = f'fraud-detector-{ns}'

        random.seed(42); np.random.seed(42); torch.manual_seed(42)
        world_size = int(os.getenv('WORLD_SIZE', '1')); rank = int(os.getenv('RANK', '0')); local_rank = int(os.getenv('LOCAL_RANK', '0'))
        distributed = world_size > 1
        if distributed: dist.init_process_group(backend='nccl' if torch.cuda.is_available() else 'gloo')

        if rank == 0: print(f'Fetching features from Feast ({feast_start_date} to {feast_end_date})...')
        from datetime import datetime
        from feast import FeatureStore, RepoConfig
        if feast_features is None:
            feast_features = ['fraud_features:distance_from_home','fraud_features:distance_from_last_transaction','fraud_features:ratio_to_median_purchase_price','fraud_features:repeat_retailer','fraud_features:used_chip','fraud_features:used_pin_number','fraud_features:online_order','fraud_features:fraud']
        config = RepoConfig(project='fraud_detection', provider='local', registry={'registry_type': 'remote', 'path': feast_registry_url}, offline_store={'type': 'remote', 'host': feast_offline_host, 'port': int(feast_offline_port)}, online_store={'type': 'sqlite', 'path': '/tmp/feast_online.db'}, entity_key_serialization_version=3)
        store = FeatureStore(config=config)
        df = store.get_historical_features(entity_df=None, features=feast_features, start_date=datetime.strptime(feast_start_date, '%Y-%m-%d'), end_date=datetime.strptime(feast_end_date, '%Y-%m-%d')).to_df()
        all_cols = [f.split(':')[-1] for f in feast_features]; feature_cols = [c for c in all_cols if c != label_column]
        df = df.dropna(subset=feature_cols + [label_column])
        train_df, val_df = train_test_split(df, test_size=val_split, random_state=42, stratify=df[label_column])
        if rank == 0: print(f'Data: {len(df)} rows, Train: {len(train_df)}, Val: {len(val_df)}')

        class TabDS(Dataset):
            def __init__(s, f, fc, lc): s.x=torch.tensor(f[fc].values, dtype=torch.float32); s.y=torch.tensor(f[lc].values, dtype=torch.float32).unsqueeze(1)
            def __len__(s): return len(s.x)
            def __getitem__(s, i): return s.x[i], s.y[i]
        class FraudMLP(nn.Module):
            def __init__(s, d, h): super().__init__(); s.net=nn.Sequential(nn.Linear(d,h),nn.ReLU(),nn.Dropout(0.2),nn.Linear(h,h//2),nn.ReLU(),nn.Linear(h//2,1))
            def forward(s, x): return s.net(x)

        device = torch.device('cpu')
        model = FraudMLP(len(feature_cols), hidden_dim).to(device)
        if distributed: model = nn.parallel.DistributedDataParallel(model)
        ts = DistributedSampler(TabDS(train_df, feature_cols, label_column)) if distributed else None
        tl = DataLoader(TabDS(train_df, feature_cols, label_column), batch_size=batch_size, sampler=ts, shuffle=(ts is None))
        vl = DataLoader(TabDS(val_df, feature_cols, label_column), batch_size=batch_size)
        crit = nn.BCEWithLogitsLoss(); opt = torch.optim.Adam(model.parameters(), lr=lr)

        for ep in range(num_epochs):
            model.train(); rl=0
            if ts: ts.set_epoch(ep)
            for xb,yb in tl: xb,yb=xb.to(device),yb.to(device); opt.zero_grad(); l=crit(model(xb),yb); l.backward(); opt.step(); rl+=l.item()
            model.eval(); tg,sc=[],[]
            with torch.no_grad():
                for xb,yb in vl: sc.extend(torch.sigmoid(model(xb)).cpu().numpy().reshape(-1).tolist()); tg.extend(yb.numpy().reshape(-1).tolist())
            auc = roc_auc_score(tg, sc) if len(set(int(v) for v in tg))>=2 else float('nan')
            if rank==0: print(f'Epoch {ep+1}/{num_epochs} | loss={rl/max(len(tl),1):.4f} | val_auc={auc:.4f}')

        if rank == 0:
            os.makedirs(output_dir, exist_ok=True); m = model.module if hasattr(model,'module') else model
            mp = os.path.join(output_dir, 'fraud_mlp_state_dict.pt'); xp = os.path.join(output_dir, 'metrics.json')
            torch.save({'model_state_dict': m.state_dict(), 'feature_columns': feature_cols, 'label_column': label_column, 'hidden_dim': hidden_dim}, mp)
            import joblib; from sklearn.neural_network import MLPClassifier
            clf = MLPClassifier(hidden_layer_sizes=(hidden_dim, hidden_dim//2), activation='relu', max_iter=1)
            clf.fit(np.zeros((2,len(feature_cols))), np.array([0,1]))
            m.eval(); clf.coefs_=[m.net[0].weight.detach().cpu().numpy().T, m.net[3].weight.detach().cpu().numpy().T, m.net[5].weight.detach().cpu().numpy().T]
            clf.intercepts_=[m.net[0].bias.detach().cpu().numpy(), m.net[3].bias.detach().cpu().numpy(), m.net[5].bias.detach().cpu().numpy()]
            jpath = os.path.join(output_dir, 'model.joblib'); joblib.dump(clf, jpath)
            y_pred=clf.predict(val_df[feature_cols].values); y_true=val_df[label_column].values.astype(int)
            metrics = {'val_auc':float(auc),'accuracy':float(accuracy_score(y_true,y_pred)),'precision':float(precision_score(y_true,y_pred,zero_division=0)),'recall':float(recall_score(y_true,y_pred,zero_division=0)),'f1_score':float(f1_score(y_true,y_pred,zero_division=0)),'confusion_matrix':confusion_matrix(y_true,y_pred).tolist(),'epochs':num_epochs,'train_rows':len(train_df),'val_rows':len(val_df),'namespace':ns}
            with open(xp,'w') as f: json.dump(metrics, f, indent=2)
            import boto3; from botocore.client import Config as BC
            s3=boto3.client('s3',endpoint_url=minio_endpoint,aws_access_key_id=minio_access_key,aws_secret_access_key=minio_secret_key,config=BC(signature_version='s3v4'),region_name='us-east-1')
            try: s3.create_bucket(Bucket=minio_bucket)
            except: pass
            s3.upload_file(mp,minio_bucket,f'{minio_model_prefix}/fraud_mlp_state_dict.pt')
            s3.upload_file(jpath,minio_bucket,f'{minio_model_prefix}/model.joblib')
            s3.upload_file(xp,minio_bucket,f'{minio_model_prefix}/metrics.json')
            print(f'Uploaded to MinIO: s3://{minio_bucket}/{minio_model_prefix}/')
        if distributed: dist.barrier(); dist.destroy_process_group()

    client = TrainerClient()
    job_name = client.train(
        trainer=CustomTrainer(
            func=train_fraud_from_feast,
            func_args={
                'num_epochs': num_epochs, 'batch_size': batch_size, 'lr': learning_rate,
                'hidden_dim': hidden_dim, 'feast_start_date': feast_start_date,
                'feast_end_date': feast_end_date, 'label_column': 'fraud',
                'val_split': val_split, 'output_dir': '/tmp/model_output',
                'minio_endpoint': minio_endpoint, 'minio_access_key': minio_access_key,
                'minio_secret_key': minio_secret_key, 'minio_bucket': minio_bucket,
                'minio_model_prefix': minio_model_prefix, 'workshop_ns': workshop_ns,
            },
            num_nodes=1,
            resources_per_node={'cpu': '2', 'memory': '4Gi'},
            packages_to_install=['torch','pandas','scikit-learn','pyarrow','boto3','joblib','feast[postgres,grpc]','psycopg2-binary','grpcio'],
        ),
        runtime='torch-distributed',
    )
    print(f'Submitted TrainJob: {job_name}')

    client.wait_for_job_status(name=job_name, status={'Running'}, timeout=600)
    print(f'{job_name} is running...')
    for line in client.get_job_logs(job_name, follow=True):
        print(line, end='')
    client.wait_for_job_status(name=job_name, timeout=60)
    print('TrainJob completed.')

### Component 2 — Quality Gate

Reads the metrics.json uploaded by the TrainJob and enforces minimum AUC.

In [ ]:
@dsl.component(
    base_image="registry.access.redhat.com/ubi9/python-311:latest",
    packages_to_install=["boto3"],
)
def quality_gate(
    minio_endpoint: str,
    minio_access_key: str,
    minio_secret_key: str,
    minio_bucket: str,
    minio_model_prefix: str,
    min_auc: float,
):
    """Read metrics.json from MinIO, enforce quality threshold."""
    import json
    import boto3
    from botocore.client import Config

    s3 = boto3.client('s3', endpoint_url=minio_endpoint, aws_access_key_id=minio_access_key,
                      aws_secret_access_key=minio_secret_key, config=Config(signature_version='s3v4'), region_name='us-east-1')

    s3.download_file(minio_bucket, f'{minio_model_prefix}/metrics.json', '/tmp/metrics.json')
    with open('/tmp/metrics.json') as f:
        metrics = json.load(f)

    auc = metrics['val_auc']
    print('=' * 60)
    print('MODEL QUALITY GATE')
    print('=' * 60)
    print(f'  ROC AUC:     {auc:.4f}  (threshold: >= {min_auc})')
    for k in ['accuracy', 'precision', 'recall', 'f1_score']:
        if k in metrics:
            print(f'  {k:12s}: {metrics[k]:.4f}')
    if 'confusion_matrix' in metrics:
        cm = metrics['confusion_matrix']
        print(f'  Confusion:   TN={cm[0][0]} FP={cm[0][1]} FN={cm[1][0]} TP={cm[1][1]}')
    print('-' * 60)
    if auc >= min_auc:
        print(f'  PASSED — AUC {auc:.4f} >= {min_auc}')
    else:
        print(f'  FAILED — AUC {auc:.4f} < {min_auc}')
        raise RuntimeError(f'Quality gate failed: AUC={auc:.4f} < {min_auc}')
    print('=' * 60)

### Component 3 — Deploy KServe InferenceService

Creates or patches a KServe InferenceService pointing at the MinIO model.

In [ ]:
@dsl.component(
    base_image="registry.access.redhat.com/ubi9/python-311:latest",
    packages_to_install=["kubernetes"],
)
def deploy_kserve(
    minio_bucket: str,
    minio_model_prefix: str,
    kserve_namespace: str,
    inference_service_name: str,
):
    """Create or patch a KServe InferenceService."""
    from kubernetes import client, config

    model_uri = f's3://{minio_bucket}/{minio_model_prefix}'
    print(f'Deploying InferenceService {inference_service_name} in {kserve_namespace}')
    print(f'Storage URI: {model_uri}')

    try: config.load_incluster_config()
    except: config.load_kube_config()

    api = client.CustomObjectsApi()
    isvc = {
        'apiVersion': 'serving.kserve.io/v1beta1', 'kind': 'InferenceService',
        'metadata': {'name': inference_service_name, 'namespace': kserve_namespace},
        'spec': {'predictor': {'serviceAccountName': 'kserve-minio-sa', 'model': {
            'modelFormat': {'name': 'sklearn'}, 'storageUri': model_uri,
            'resources': {'requests': {'memory': '512Mi', 'cpu': '250m'}, 'limits': {'memory': '1Gi', 'cpu': '500m'}}
        }}}
    }
    try:
        api.get_namespaced_custom_object(group='serving.kserve.io', version='v1beta1', namespace=kserve_namespace, plural='inferenceservices', name=inference_service_name)
        api.patch_namespaced_custom_object(group='serving.kserve.io', version='v1beta1', namespace=kserve_namespace, plural='inferenceservices', name=inference_service_name, body=isvc)
        print(f'Patched existing InferenceService')
    except client.exceptions.ApiException as e:
        if e.status == 404:
            api.create_namespaced_custom_object(group='serving.kserve.io', version='v1beta1', namespace=kserve_namespace, plural='inferenceservices', body=isvc)
            print(f'Created InferenceService')
        else: raise

## 3) Define the pipeline

The pipeline submits a TrainerV2 TrainJob (which fetches from Feast, trains, uploads to MinIO),
then checks quality, and optionally deploys to KServe.

In [ ]:
@dsl.pipeline(
    name='Fraud Detection — TrainerV2 + Feast + KServe',
    description='Submits a TrainerV2 TrainJob that fetches from Feast, trains FraudMLP, uploads to MinIO. Then quality gate + KServe deploy.',
)
def fraud_pipeline(
    workshop_ns: str = 'mlops-workshop',
    feast_start_date: str = '2025-01-01',
    feast_end_date: str = '2025-03-31',
    num_epochs: int = 5,
    batch_size: int = 256,
    learning_rate: float = 0.001,
    hidden_dim: int = 64,
    val_split: float = 0.2,
    minio_endpoint: str = 'http://minio-service.kubeflow.svc.cluster.local:9000',
    minio_access_key: str = 'minio',
    minio_secret_key: str = 'minio123',
    minio_bucket: str = 'models',
    minio_model_prefix: str = 'fraud-detector',
    min_auc: float = 0.7,
    deploy_model: bool = True,
    inference_service_name: str = 'fraud-detector',
):
    train_task = submit_train_job(
        workshop_ns=workshop_ns,
        feast_start_date=feast_start_date, feast_end_date=feast_end_date,
        num_epochs=num_epochs, batch_size=batch_size,
        learning_rate=learning_rate, hidden_dim=hidden_dim, val_split=val_split,
        minio_endpoint=minio_endpoint, minio_access_key=minio_access_key,
        minio_secret_key=minio_secret_key, minio_bucket=minio_bucket,
        minio_model_prefix=minio_model_prefix,
    ).set_display_name('1. Submit TrainerV2 TrainJob')

    gate_task = quality_gate(
        minio_endpoint=minio_endpoint, minio_access_key=minio_access_key,
        minio_secret_key=minio_secret_key, minio_bucket=minio_bucket,
        minio_model_prefix=minio_model_prefix, min_auc=min_auc,
    ).after(train_task).set_display_name('2. Quality Gate')

    with dsl.If(deploy_model == True, name='Deploy if Approved'):
        deploy_kserve(
            minio_bucket=minio_bucket, minio_model_prefix=minio_model_prefix,
            kserve_namespace=workshop_ns, inference_service_name=inference_service_name,
        ).after(gate_task).set_display_name('3. Deploy KServe')

print('Pipeline defined.')

## 4) Compile pipeline

In [ ]:
from kfp import compiler

YAML_PATH = 'fraud_pipeline_trainerv2.yaml'
compiler.Compiler().compile(pipeline_func=fraud_pipeline, package_path=YAML_PATH)
print(f'Compiled -> {YAML_PATH}')

## 5) Submit pipeline run

In [ ]:
import os
import kfp

KFP_ENDPOINT = 'http://kfp-ui-kubeflow.apps.rosa.k2s7v9j3f3g5j9l.yqif.p3.openshiftapps.com'
EXPERIMENT_NAME = 'fraud-detection-trainerv2'
RUN_NAME = 'fraud-trainerv2-run-01'

MY_NS = os.environ.get('WORKSHOP_NS', 'mlops-workshop')

kfp_client = kfp.Client(host=KFP_ENDPOINT)
print(f'Connected to KFP at {KFP_ENDPOINT}')

In [ ]:
PIPELINE_PARAMS = {
    'workshop_ns': MY_NS,
    'feast_start_date': '2025-01-01',
    'feast_end_date': '2025-03-31',
    'num_epochs': 5,
    'batch_size': 256,
    'learning_rate': 0.001,
    'hidden_dim': 64,
    'val_split': 0.2,
    'minio_endpoint': 'http://minio-service.kubeflow.svc.cluster.local:9000',
    'minio_access_key': 'minio',
    'minio_secret_key': 'minio123',
    'minio_bucket': 'models',
    'minio_model_prefix': f'fraud-detector-{MY_NS}',
    'min_auc': 0.7,
    'deploy_model': True,
    'inference_service_name': 'fraud-detector',
}

print('Pipeline parameters:')
for k, v in PIPELINE_PARAMS.items():
    print(f'  {k:30s} = {v}')

In [ ]:
run = kfp_client.create_run_from_pipeline_package(
    pipeline_file=YAML_PATH,
    arguments=PIPELINE_PARAMS,
    run_name=RUN_NAME,
    experiment_name=EXPERIMENT_NAME,
)
print(f'Run submitted: {run.run_id}')
print(f'View in UI: {KFP_ENDPOINT}/#/runs/details/{run.run_id}')

## 6) Monitor run

In [ ]:
kfp_client.wait_for_run_completion(run_id=run.run_id, timeout=900)
print('Pipeline run completed.')